**Première partie**

**Modèle :** Transformer

**Dataset :** OASIS-2


**Réalisé par :** Sameh BEN HAMIDA


In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
# Assurer que venv_mia_tf est utilise
import sys
import os

# Ajouter le chemin du venv_mia_tf au debut du sys.path (portable)
base_dir = os.getcwd()
venv_site_packages = os.path.join(base_dir, 'venv_mia_tf', 'Lib', 'site-packages')
if venv_site_packages not in sys.path and os.path.isdir(venv_site_packages):
    sys.path.insert(0, venv_site_packages)

# Verifier les versions
print("=" * 60)
print("VERIFICATION DE L'ENVIRONNEMENT VIRTUEL")
print("=" * 60)
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

try:
    import tensorflow as tf
    import tensorflow_privacy
    import numpy as np
    import pandas as pd
    import sklearn

    print(f"TensorFlow version: {tf.__version__}")
    print(f"TensorFlow Privacy version: {tensorflow_privacy.__version__}")
    print(f"NumPy version: {np.__version__}")
    print(f"Pandas version: {pd.__version__}")
    print(f"Scikit-learn version: {sklearn.__version__}")
    print("=" * 60)
    print("ENVIRONNEMENT PRET")
    print("=" * 60)
except Exception as e:
    print(f"Erreur d'import: {e}")
    print("Verifier que venv_mia_tf est correctement configure")
    sys.exit(1)

VERIFICATION DE L'ENVIRONNEMENT VIRTUEL
Python executable: c:\Users\khalil\Desktop\mia test - Copie\venv_mia_tf\Scripts\python.exe
Python version: 3.9.13 (tags/v3.9.13:6de2ca5, May 17 2022, 16:36:42) [MSC v.1929 64 bit (AMD64)]
TensorFlow version: 2.12.0
TensorFlow Privacy version: 0.9.0
NumPy version: 1.23.5
Pandas version: 1.5.3
Scikit-learn version: 1.2.2
ENVIRONNEMENT PRET


## Pipeline final nettoye (Transformer + 3 attaques MIA)

Cette section garde uniquement l'essentiel:
1. Entrainer un Transformer binaire sur OASIS.
2. Afficher les metriques du Transformer.
3. Lancer 3 attaques MIA uniquement: threshold_loss, logistic, shadow_meta.
4. Conserver une seule attaque Shadow Model efficace (shadow_meta) et exclure les autres variantes Shadow.
5. Afficher les metriques comparatives finales.

In [4]:
import os
import random
import numpy as np
import pandas as pd

# Determinism settings should be set before/with TensorFlow import.
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TF_DETERMINISTIC_OPS"] = "1"

import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from tensorflow.keras import layers, Model


def set_global_determinism(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


# -----------------------------
# 1) Chargement + nettoyage + split
# -----------------------------
SEED = 42
set_global_determinism(SEED)

FEATURE_COLS = ["MR Delay", "Age", "EDUC", "SES", "MMSE", "CDR", "eTIV", "nWBV", "ASF"]
TARGET_COL = "Group"

df = pd.read_csv("oasis_longitudinal.csv")
print("=== Stats brutes ===")
print(f"Shape brute: {df.shape}")
print(df[FEATURE_COLS + [TARGET_COL]].isna().sum())

df = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).copy()
df[TARGET_COL] = (df[TARGET_COL] == "Demented").astype(int)

X = df[FEATURE_COLS].values.astype(np.float32)
y = df[TARGET_COL].values.astype(np.int32)

print("\n=== Stats après nettoyage ===")
print(f"Shape nettoyée: {X.shape}")
print(f"Classe 0: {(y == 0).sum()} | Classe 1: {(y == 1).sum()}")
print(f"Ratio classe 1: {y.mean():.4f}")

# Configuration finale retenue (cas reel)
TEST_SIZE = 0.80
EPOCHS = 260
BATCH_SIZE = 32
DROPOUT = 0.00

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

X_train_seq = X_train.reshape(-1, 1, X_train.shape[1])
X_test_seq = X_test.reshape(-1, 1, X_test.shape[1])

print("\n=== Stats split ===")
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print(f"Train classe 1 ratio: {y_train.mean():.4f}")
print(f"Test classe 1 ratio: {y_test.mean():.4f}")


def build_transformer(n_features, d_model=64, n_heads=4, ff_dim=128, dropout=0.0):
    inp = layers.Input(shape=(1, n_features))

    x = layers.Dense(d_model)(inp)
    attn = layers.MultiHeadAttention(num_heads=n_heads, key_dim=max(1, d_model // n_heads), dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn)

    ff = layers.Dense(ff_dim, activation="relu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(d_model)(ff)
    x = layers.LayerNormalization(epsilon=1e-6)(x + ff)

    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


def mia_features(proba, y_true):
    eps = 1e-8
    p = np.clip(proba, eps, 1.0 - eps)
    loss = -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
    conf = np.maximum(p, 1 - p)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    cor = ((p >= 0.5).astype(int) == y_true).astype(float)
    margin = np.abs(p - 0.5)
    return np.column_stack([loss, conf, ent, cor, margin])


def attack_row(name, y_true, y_pred, y_score):
    return {
        "attack": name,
        "auc": roc_auc_score(y_true, y_score),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

=== Stats brutes ===
Shape brute: (373, 15)
MR Delay     0
Age          0
EDUC         0
SES         19
MMSE         2
CDR          0
eTIV         0
nWBV         0
ASF          0
Group        0
dtype: int64

=== Stats après nettoyage ===
Shape nettoyée: (354, 9)
Classe 0: 227 | Classe 1: 127
Ratio classe 1: 0.3588

=== Stats split ===
Train size: 70 | Test size: 284
Train classe 1 ratio: 0.3571
Test classe 1 ratio: 0.3592


In [5]:
# 2) Entraînement Transformer + validation (anti-overfit)

from tensorflow.keras.callbacks import EarlyStopping

# Modèle normal (métriques réalistes)
tf.keras.backend.clear_session()
set_global_determinism(SEED + 10)
target_model = build_transformer(n_features=X_train.shape[1], dropout=0.15)

es = EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True,
    verbose=0,
)

history = target_model.fit(
    X_train_seq,
    y_train,
    epochs=120,
    batch_size=32,
    validation_split=0.2,
    callbacks=[es],
    verbose=0,
)

p_train = target_model.predict(X_train_seq, verbose=0).ravel()
p_test = target_model.predict(X_test_seq, verbose=0).ravel()

train_acc = accuracy_score(y_train, (p_train >= 0.5).astype(int))
test_acc = accuracy_score(y_test, (p_test >= 0.5).astype(int))
train_auc = roc_auc_score(y_train, p_train)
test_auc = roc_auc_score(y_test, p_test)

best_val_acc = max(history.history["val_accuracy"]) if "val_accuracy" in history.history else float("nan")
best_val_loss = min(history.history["val_loss"]) if "val_loss" in history.history else float("nan")

transformer_metrics = pd.DataFrame(
    [{
        "train_acc": train_acc,
        "test_acc": test_acc,
        "gap": train_acc - test_acc,
        "train_auc": train_auc,
        "test_auc": test_auc,
        "best_val_acc": best_val_acc,
        "best_val_loss": best_val_loss,
        "epochs_ran": len(history.history["loss"]),
    }]
)

print("=== Métriques Transformer (avec validation) ===")
print(f"Dernière loss train: {history.history['loss'][-1]:.6f}")
print(f"Meilleure val_acc: {best_val_acc:.4f} | Meilleure val_loss: {best_val_loss:.6f}")
display(transformer_metrics.round(4))

# Modèle stress-test MIA (plus vulnérable pour augmenter la puissance de l'attaque Shadow)
# On garde les métriques normales ci-dessus pour l'évaluation clinique.
tf.keras.backend.clear_session()
set_global_determinism(SEED + 20)
mia_target_model = build_transformer(n_features=X_train.shape[1], dropout=0.0)
mia_target_model.fit(
    X_train_seq,
    y_train,
    epochs=260,
    batch_size=32,
    verbose=0,
)

p_train_mia = mia_target_model.predict(X_train_seq, verbose=0).ravel()
p_test_mia = mia_target_model.predict(X_test_seq, verbose=0).ravel()

print("\n=== Modèle cible MIA (stress-test) ===")
print(f"train_acc_mia={accuracy_score(y_train, (p_train_mia >= 0.5).astype(int)):.4f}")
print(f"test_acc_mia={accuracy_score(y_test, (p_test_mia >= 0.5).astype(int)):.4f}")

=== Métriques Transformer (avec validation) ===
Dernière loss train: 0.029273
Meilleure val_acc: 1.0000 | Meilleure val_loss: 0.204655


,train_acc,test_acc,gap,train_auc,test_auc,best_val_acc,best_val_loss,epochs_ran
0,0.9714,0.9331,0.0383,0.9956,0.9892,1.0,0.2047,18



=== Modèle cible MIA (stress-test) ===
train_acc_mia=1.0000
test_acc_mia=0.9331


In [6]:
# 3) Features MIA + stats (depuis le modèle cible MIA stress-test)

F_mem = mia_features(p_train_mia, y_train)
F_non = mia_features(p_test_mia, y_test)
X_attack = np.vstack([F_mem, F_non])
y_attack = np.concatenate([
    np.ones(len(F_mem), dtype=int),
    np.zeros(len(F_non), dtype=int),
])

Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(
    X_attack, y_attack, test_size=0.4, random_state=SEED, stratify=y_attack
)

print("=== Stats Features MIA ===")
print(f"F_mem: {F_mem.shape} | F_non: {F_non.shape}")
print(f"X_attack: {X_attack.shape}")
print(f"Attack train: {Xa_tr.shape} | Attack test: {Xa_te.shape}")
print(f"Loss mean members: {F_mem[:,0].mean():.6f}")
print(f"Loss mean non-members: {F_non[:,0].mean():.6f}")

=== Stats Features MIA ===
F_mem: (70, 5) | F_non: (284, 5)
X_attack: (354, 5)
Attack train: (212, 5) | Attack test: (142, 5)
Loss mean members: 0.000092
Loss mean non-members: 0.367018


In [7]:
# 4) Attaque 1: Threshold loss + stats détaillées

score_tr = -Xa_tr[:, 0]
score_te = -Xa_te[:, 0]

cand = np.linspace(score_tr.min(), score_tr.max(), 400)
best_thr = cand[0]
best_bal = -1.0

for t in cand:
    pred = (score_tr >= t).astype(int)
    tpr = ((pred == 1) & (ya_tr == 1)).sum() / max((ya_tr == 1).sum(), 1)
    tnr = ((pred == 0) & (ya_tr == 0)).sum() / max((ya_tr == 0).sum(), 1)
    bal = 0.5 * (tpr + tnr)
    if bal > best_bal:
        best_bal = bal
        best_thr = t

pred_thr = (score_te >= best_thr).astype(int)
thr_metrics = attack_row("threshold_loss", ya_te, pred_thr, score_te)

print("=== Attaque Threshold Loss ===")
print(f"Best threshold: {best_thr:.6f}")
print(f"Balanced accuracy (train attack): {best_bal:.4f}")
print(pd.DataFrame([thr_metrics]).round(4))

=== Attaque Threshold Loss ===
Best threshold: -0.023456
Balanced accuracy (train attack): 0.5353
           attack     auc  accuracy  precision  recall      f1
0  threshold_loss  0.4821    0.3099     0.2222     1.0  0.3636


In [8]:
# 5) Attaque 2: Logistic + stats détaillées

lr_attack = LogisticRegression(max_iter=1000, random_state=SEED)
lr_attack.fit(Xa_tr, ya_tr)
proba_lr = lr_attack.predict_proba(Xa_te)[:, 1]
pred_lr = (proba_lr >= 0.5).astype(int)

lr_metrics = attack_row("logistic", ya_te, pred_lr, proba_lr)

print("=== Attaque Logistic ===")
print(f"Coefficients: {lr_attack.coef_.shape}")
print(pd.DataFrame([lr_metrics]).round(4))

=== Attaque Logistic ===


Coefficients: (1, 5)
     attack     auc  accuracy  precision  recall   f1
0  logistic  0.4818    0.8028        0.0     0.0  0.0


In [9]:
# 6) Attaque 3: Shadow Meta + LiRA-like + Boundary (black-box, sans fuite, selection interne)

from sklearn.metrics import roc_curve


def mia_features_enriched(proba, y_true, eps=1e-6):
    p = np.clip(np.asarray(proba, dtype=np.float64), eps, 1.0 - eps)
    y_true = np.asarray(y_true, dtype=np.int32)

    loss = -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
    conf = np.maximum(p, 1 - p)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    cor = ((p >= 0.5).astype(np.int32) == y_true).astype(np.float64)
    margin = np.abs(p - 0.5)

    logit = np.log(p / (1 - p))
    p_true = np.where(y_true == 1, p, 1 - p)
    p_false = 1.0 - p_true

    feats = np.column_stack([loss, conf, ent, cor, margin, p, logit, p_true, p_false])
    return np.nan_to_num(feats, nan=0.0, posinf=1e6, neginf=-1e6)


def tpr_at_fpr(y_true, y_score, target_fpr=0.01):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    idx = np.searchsorted(fpr, target_fpr, side='right') - 1
    idx = np.clip(idx, 0, len(tpr) - 1)
    return float(tpr[idx])


def rank01(x):
    x = np.asarray(x)
    order = np.argsort(np.argsort(x))
    if len(x) <= 1:
        return np.zeros_like(x, dtype=np.float64)
    return order.astype(np.float64) / float(len(x) - 1)


def normal_logpdf(x, mu, sigma):
    sigma = max(float(sigma), 1e-3)
    z = (x - mu) / sigma
    return -0.5 * (np.log(2.0 * np.pi) + 2.0 * np.log(sigma) + z * z)


def predict_label_blackbox(model, X_2d):
    X_seq = X_2d.reshape(-1, 1, X_2d.shape[1])
    p = model.predict(X_seq, verbose=0).ravel()
    return (p >= 0.5).astype(np.int32)


def boundary_distance_label_only(model, x, pred_label, seed, max_alpha=2.0, n_dirs=24, n_steps=8):
    rng = np.random.default_rng(seed)
    d = rng.normal(size=(n_dirs, x.shape[0]))
    d = d / (np.linalg.norm(d, axis=1, keepdims=True) + 1e-12)

    alphas = np.geomspace(0.02, max_alpha, num=n_steps)
    candidates = []
    for i in range(n_dirs):
        for a in alphas:
            candidates.append(x + a * d[i])
    candidates = np.asarray(candidates, dtype=np.float32)

    labels = predict_label_blackbox(model, candidates)
    labels = labels.reshape(n_dirs, n_steps)
    flips = labels != int(pred_label)

    dist = max_alpha * 1.5
    for i in range(n_dirs):
        where = np.where(flips[i])[0]
        if len(where) > 0:
            cand = float(alphas[int(where[0])])
            if cand < dist:
                dist = cand
    return dist


# ---- 1) Split strict A/B/C ----
X_ab, X_aux, y_ab, y_aux = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_target_mem, X_target_non, y_target_mem, y_target_non = train_test_split(
    X_ab, y_ab, test_size=0.45, random_state=SEED + 1, stratify=y_ab
)

sc_t = StandardScaler()
X_target_mem_sc = sc_t.fit_transform(X_target_mem).astype(np.float32)
X_target_non_sc = sc_t.transform(X_target_non).astype(np.float32)
X_target_mem_seq = X_target_mem_sc.reshape(-1, 1, X_target_mem_sc.shape[1])
X_target_non_seq = X_target_non_sc.reshape(-1, 1, X_target_non_sc.shape[1])

tf.keras.backend.clear_session()
set_global_determinism(SEED + 1000)
target_shadow_eval_model = build_transformer(n_features=X_target_mem_sc.shape[1], dropout=0.0)
target_shadow_eval_model.fit(X_target_mem_seq, y_target_mem, epochs=220, batch_size=16, verbose=0)

p_mem_target = target_shadow_eval_model.predict(X_target_mem_seq, verbose=0).ravel()
p_non_target = target_shadow_eval_model.predict(X_target_non_seq, verbose=0).ravel()

F_mem_target = mia_features_enriched(p_mem_target, y_target_mem)
F_non_target = mia_features_enriched(p_non_target, y_target_non)
X_attack_target = np.vstack([F_mem_target, F_non_target])
y_attack_target = np.concatenate([
    np.ones(len(F_mem_target), dtype=int),
    np.zeros(len(F_non_target), dtype=int),
])
y_attack_target_class = np.concatenate([
    y_target_mem.astype(int),
    y_target_non.astype(int),
])

X_query_target = np.vstack([X_target_mem_sc, X_target_non_sc])
y_pred_target = np.concatenate([
    (p_mem_target >= 0.5).astype(np.int32),
    (p_non_target >= 0.5).astype(np.int32),
])

# ---- 2) Shadow models sur C uniquement ----
shadow_features = []
shadow_labels = []
shadow_cls = []
shadow_in_scores = {0: [], 1: []}
shadow_out_scores = {0: [], 1: []}

n_shadows = 50
for s in range(n_shadows):
    xs_mem, xs_non, ys_mem, ys_non = train_test_split(
        X_aux, y_aux, test_size=0.5, random_state=SEED + 100 + s, stratify=y_aux
    )

    sc_s = StandardScaler()
    xs_mem_sc = sc_s.fit_transform(xs_mem).astype(np.float32)
    xs_non_sc = sc_s.transform(xs_non).astype(np.float32)

    xs_mem_seq = xs_mem_sc.reshape(-1, 1, xs_mem_sc.shape[1])
    xs_non_seq = xs_non_sc.reshape(-1, 1, xs_non_sc.shape[1])

    tf.keras.backend.clear_session()
    set_global_determinism(SEED + 2000 + s)
    sh_model = build_transformer(n_features=xs_mem_sc.shape[1], dropout=0.0)
    sh_model.fit(xs_mem_seq, ys_mem, epochs=140, batch_size=16, verbose=0)

    ps_mem = sh_model.predict(xs_mem_seq, verbose=0).ravel()
    ps_non = sh_model.predict(xs_non_seq, verbose=0).ravel()

    fm = mia_features_enriched(ps_mem, ys_mem)
    fn = mia_features_enriched(ps_non, ys_non)
    shadow_features.append(np.vstack([fm, fn]))
    shadow_labels.append(np.concatenate([
        np.ones(len(fm), dtype=int),
        np.zeros(len(fn), dtype=int),
    ]))
    shadow_cls.append(np.concatenate([
        ys_mem.astype(int),
        ys_non.astype(int),
    ]))

    p_true_mem = np.where(ys_mem == 1, ps_mem, 1.0 - ps_mem)
    p_true_non = np.where(ys_non == 1, ps_non, 1.0 - ps_non)
    s_mem = np.log(np.clip(p_true_mem, 1e-6, 1 - 1e-6) / np.clip(1.0 - p_true_mem, 1e-6, 1 - 1e-6))
    s_non = np.log(np.clip(p_true_non, 1e-6, 1 - 1e-6) / np.clip(1.0 - p_true_non, 1e-6, 1 - 1e-6))

    for cls in [0, 1]:
        shadow_in_scores[cls].extend(s_mem[ys_mem == cls].tolist())
        shadow_out_scores[cls].extend(s_non[ys_non == cls].tolist())

X_shadow = np.vstack(shadow_features)
y_shadow = np.concatenate(shadow_labels)
y_shadow_cls = np.concatenate(shadow_cls)

lira_params = {}
for cls in [0, 1]:
    arr_in = np.asarray(shadow_in_scores[cls], dtype=np.float64)
    arr_out = np.asarray(shadow_out_scores[cls], dtype=np.float64)
    lira_params[cls] = {
        'mu_in': float(arr_in.mean()) if len(arr_in) else 0.0,
        'sd_in': float(arr_in.std(ddof=1)) if len(arr_in) > 1 else 1.0,
        'mu_out': float(arr_out.mean()) if len(arr_out) else 0.0,
        'sd_out': float(arr_out.std(ddof=1)) if len(arr_out) > 1 else 1.0,
    }

# Score boundary black-box calcule une fois
boundary_raw = np.zeros(len(X_query_target), dtype=np.float64)
for i in range(len(X_query_target)):
    boundary_raw[i] = boundary_distance_label_only(
        target_shadow_eval_model,
        X_query_target[i],
        y_pred_target[i],
        seed=SEED + 9000 + i,
        max_alpha=2.0,
        n_dirs=24,
        n_steps=8,
    )
score_boundary_all = rank01(boundary_raw)


# ---- 3) Eval attack: selection de score sur validation interne uniquement ----
seed_list = [11, 22, 33, 44, 55]
rows_seed = []

for s_eval in seed_list:
    Xa_tr_full, Xa_te, ya_tr_full, ya_te, yc_tr_full, yc_te, sb_tr_full, sb_te = train_test_split(
        X_attack_target,
        y_attack_target,
        y_attack_target_class,
        score_boundary_all,
        test_size=0.4,
        random_state=s_eval,
        stratify=y_attack_target,
    )

    # Validation interne pour choisir le score (jamais le holdout Xa_te)
    Xa_tr, Xa_val, ya_tr, ya_val, yc_tr, yc_val, sb_tr, sb_val = train_test_split(
        Xa_tr_full,
        ya_tr_full,
        yc_tr_full,
        sb_tr_full,
        test_size=0.25,
        random_state=s_eval + 99,
        stratify=ya_tr_full,
    )

    X_meta_train = np.vstack([X_shadow, Xa_tr])
    y_meta_train = np.concatenate([y_shadow, ya_tr])
    yc_meta_train = np.concatenate([y_shadow_cls, yc_tr])

    models_by_cls = {}
    for cls in [0, 1]:
        msk = yc_meta_train == cls
        if msk.sum() >= 20 and len(np.unique(y_meta_train[msk])) == 2:
            m_cls = GradientBoostingClassifier(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=3,
                random_state=SEED + cls + s_eval,
            )
            m_cls.fit(X_meta_train[msk], y_meta_train[msk])
            models_by_cls[cls] = m_cls

    global_meta = GradientBoostingClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=3,
        random_state=SEED + s_eval,
    )
    global_meta.fit(X_meta_train, y_meta_train)

    def score_meta_fn(Xa, yc):
        out = np.zeros(len(Xa), dtype=np.float64)
        for i in range(len(Xa)):
            cls = int(yc[i])
            model_i = models_by_cls.get(cls, global_meta)
            out[i] = model_i.predict_proba(Xa[i:i+1])[:, 1][0]
        return out

    def score_lira_fn(Xa, yc):
        p_true = np.clip(Xa[:, 7], 1e-6, 1 - 1e-6)
        s_vals = np.log(p_true / (1.0 - p_true))
        out = np.zeros(len(Xa), dtype=np.float64)
        for i in range(len(Xa)):
            prm = lira_params[int(yc[i])]
            li = normal_logpdf(s_vals[i], prm['mu_in'], prm['sd_in'])
            lo = normal_logpdf(s_vals[i], prm['mu_out'], prm['sd_out'])
            out[i] = li - lo
        return out

    score_meta_val = score_meta_fn(Xa_val, yc_val)
    score_lira_val = score_lira_fn(Xa_val, yc_val)
    score_meta_te = score_meta_fn(Xa_te, yc_te)
    score_lira_te = score_lira_fn(Xa_te, yc_te)

    candidates = {
        'meta_only': rank01(score_meta_val),
        'lira_only': rank01(score_lira_val),
        'boundary_only': rank01(sb_val),
        'fusion_0.45_0.25_0.30': 0.45 * rank01(score_meta_val) + 0.25 * rank01(score_lira_val) + 0.30 * rank01(sb_val),
        'fusion_0.35_0.45_0.20': 0.35 * rank01(score_meta_val) + 0.45 * rank01(score_lira_val) + 0.20 * rank01(sb_val),
        'fusion_0.30_0.30_0.40': 0.30 * rank01(score_meta_val) + 0.30 * rank01(score_lira_val) + 0.40 * rank01(sb_val),
        'fusion_0.40_0.35_0.25': 0.40 * rank01(score_meta_val) + 0.35 * rank01(score_lira_val) + 0.25 * rank01(sb_val),
        'fusion_0.50_0.30_0.20': 0.50 * rank01(score_meta_val) + 0.30 * rank01(score_lira_val) + 0.20 * rank01(sb_val),
        'fusion_0.25_0.50_0.25': 0.25 * rank01(score_meta_val) + 0.50 * rank01(score_lira_val) + 0.25 * rank01(sb_val),
    }

    best_name = None
    best_auc_val = -1.0
    for name, sc in candidates.items():
        a = roc_auc_score(ya_val, sc)
        if a > best_auc_val:
            best_auc_val = a
            best_name = name

    # Appliquer le score choisi sur holdout final uniquement
    if best_name == 'meta_only':
        score_fusion = rank01(score_meta_te)
    elif best_name == 'lira_only':
        score_fusion = rank01(score_lira_te)
    elif best_name == 'boundary_only':
        score_fusion = rank01(sb_te)
    elif best_name == 'fusion_0.45_0.25_0.30':
        score_fusion = 0.45 * rank01(score_meta_te) + 0.25 * rank01(score_lira_te) + 0.30 * rank01(sb_te)
    elif best_name == 'fusion_0.35_0.45_0.20':
        score_fusion = 0.35 * rank01(score_meta_te) + 0.45 * rank01(score_lira_te) + 0.20 * rank01(sb_te)
    else:
        score_fusion = 0.30 * rank01(score_meta_te) + 0.30 * rank01(score_lira_te) + 0.40 * rank01(sb_te)

    pred_shadow = (score_fusion >= 0.5).astype(int)

    row = attack_row('shadow_meta', ya_te, pred_shadow, score_fusion)
    row['seed'] = s_eval
    row['selected_scorer'] = best_name
    row['selected_val_auc'] = float(best_auc_val)
    row['auc_meta_only'] = roc_auc_score(ya_te, score_meta_te)
    row['auc_lira_only'] = roc_auc_score(ya_te, score_lira_te)
    row['auc_boundary_only'] = roc_auc_score(ya_te, sb_te)
    row['tpr_at_1pct_fpr'] = tpr_at_fpr(ya_te, score_fusion, 0.01)
    row['tpr_at_5pct_fpr'] = tpr_at_fpr(ya_te, score_fusion, 0.05)
    rows_seed.append(row)

shadow_seed_df = pd.DataFrame(rows_seed)

shadow_metrics = {
    'attack': 'shadow_meta',
    'auc': float(shadow_seed_df['auc'].mean()),
    'accuracy': float(shadow_seed_df['accuracy'].mean()),
    'precision': float(shadow_seed_df['precision'].mean()),
    'recall': float(shadow_seed_df['recall'].mean()),
    'f1': float(shadow_seed_df['f1'].mean()),
}

print('=== Attaque Shadow Meta + LiRA-like + Boundary (sans fuite, selection interne) ===')
print(f"Split sizes | target_mem(A): {X_target_mem.shape} | target_non(B): {X_target_non.shape} | aux(C): {X_aux.shape}")
print(f"X_shadow: {X_shadow.shape} | n_shadows: {n_shadows} | features: {X_shadow.shape[1]}")
print('Moyenne sur 5 seeds (score choisi par validation interne):')
print(pd.DataFrame([shadow_metrics]).round(4))
print(f"AUC meta-only mean: {shadow_seed_df['auc_meta_only'].mean():.4f}")
print(f"AUC lira-only mean: {shadow_seed_df['auc_lira_only'].mean():.4f}")
print(f"AUC boundary-only mean: {shadow_seed_df['auc_boundary_only'].mean():.4f}")

print('\nScore choisi par seed:')
print(shadow_seed_df[['seed', 'selected_scorer', 'selected_val_auc']].round(4).to_string(index=False))

shadow_seed_summary = pd.DataFrame([
    {
        'auc_mean': shadow_seed_df['auc'].mean(),
        'auc_std': shadow_seed_df['auc'].std(ddof=1),
        'acc_mean': shadow_seed_df['accuracy'].mean(),
        'acc_std': shadow_seed_df['accuracy'].std(ddof=1),
        'tpr_at_1pct_fpr_mean': shadow_seed_df['tpr_at_1pct_fpr'].mean(),
        'tpr_at_1pct_fpr_std': shadow_seed_df['tpr_at_1pct_fpr'].std(ddof=1),
        'tpr_at_5pct_fpr_mean': shadow_seed_df['tpr_at_5pct_fpr'].mean(),
        'tpr_at_5pct_fpr_std': shadow_seed_df['tpr_at_5pct_fpr'].std(ddof=1),
    }
])

print('\nDetails par seed:')
display(shadow_seed_df[['seed', 'auc', 'auc_meta_only', 'auc_lira_only', 'auc_boundary_only', 'accuracy', 'precision', 'recall', 'f1', 'tpr_at_1pct_fpr', 'tpr_at_5pct_fpr']].round(4))
print('\nResume robuste (mean +- std):')
display(shadow_seed_summary.round(4))

=== Attaque Shadow Meta + LiRA-like + Boundary (sans fuite, selection interne) ===
Split sizes | target_mem(A): (135, 9) | target_non(B): (112, 9) | aux(C): (107, 9)
X_shadow: (5350, 9) | n_shadows: 50 | features: 9
Moyenne sur 5 seeds (score choisi par validation interne):
        attack     auc  accuracy  precision  recall      f1
0  shadow_meta  0.6151    0.6485     0.6807  0.6815  0.6802
AUC meta-only mean: 0.5732
AUC lira-only mean: 0.5802
AUC boundary-only mean: 0.6281

Score choisi par seed:
 seed       selected_scorer  selected_val_auc
   11 fusion_0.25_0.50_0.25            0.8235
   22         boundary_only            0.7000
   33 fusion_0.30_0.30_0.40            0.6691
   44 fusion_0.25_0.50_0.25            0.7706
   55         boundary_only            0.6765

Details par seed:


,seed,auc,auc_meta_only,auc_lira_only,auc_boundary_only,accuracy,precision,recall,f1,tpr_at_1pct_fpr,tpr_at_5pct_fpr
0,11,0.6121,0.5852,0.5969,0.5770,0.5859,0.6140,0.6481,0.6306,0.0000,0.1111
1,22,0.6198,0.6284,0.5949,0.6198,0.7374,0.7800,0.7222,0.7500,0.0185,0.0556
2,33,0.5780,0.5010,0.5377,0.6395,0.6061,0.6364,0.6481,0.6422,0.0000,0.1111
3,44,0.6447,0.5735,0.5424,0.6835,0.6162,0.6333,0.7037,0.6667,0.1111,0.1111
4,55,0.6210,0.5780,0.6292,0.6210,0.6970,0.7400,0.6852,0.7115,0.0185,0.0556



Resume robuste (mean +- std):


,auc_mean,auc_std,acc_mean,acc_std,tpr_at_1pct_fpr_mean,tpr_at_1pct_fpr_std,tpr_at_5pct_fpr_mean,tpr_at_5pct_fpr_std
0,0.6151,0.0241,0.6485,0.0652,0.0296,0.0465,0.0889,0.0304


In [10]:
# 6-bis) Exemples de données "volées correctement" par Shadow (portion seulement)

# On reconstruit une évaluation holdout sur un seed fixe pour afficher des exemples lisibles.
seed_show = 55
show_n = 10

idx_all = np.arange(len(X_attack_target))
X_query_target_raw = np.vstack([X_target_mem, X_target_non]).astype(np.float32)

Xa_tr_full_s, Xa_te_s, ya_tr_full_s, ya_te_s, yc_tr_full_s, yc_te_s, sb_tr_full_s, sb_te_s, idx_tr_full_s, idx_te_s = train_test_split(
    X_attack_target,
    y_attack_target,
    y_attack_target_class,
    score_boundary_all,
    idx_all,
    test_size=0.4,
    random_state=seed_show,
    stratify=y_attack_target,
)

Xa_tr_s, Xa_val_s, ya_tr_s, ya_val_s, yc_tr_s, yc_val_s, sb_tr_s, sb_val_s, idx_tr_s, idx_val_s = train_test_split(
    Xa_tr_full_s,
    ya_tr_full_s,
    yc_tr_full_s,
    sb_tr_full_s,
    idx_tr_full_s,
    test_size=0.25,
    random_state=seed_show + 99,
    stratify=ya_tr_full_s,
)

X_meta_train_s = np.vstack([X_shadow, Xa_tr_s])
y_meta_train_s = np.concatenate([y_shadow, ya_tr_s])
yc_meta_train_s = np.concatenate([y_shadow_cls, yc_tr_s])

models_by_cls_s = {}
for cls in [0, 1]:
    msk = yc_meta_train_s == cls
    if msk.sum() >= 20 and len(np.unique(y_meta_train_s[msk])) == 2:
        m_cls_s = GradientBoostingClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=3,
            random_state=SEED + cls + seed_show,
        )
        m_cls_s.fit(X_meta_train_s[msk], y_meta_train_s[msk])
        models_by_cls_s[cls] = m_cls_s

global_meta_s = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    random_state=SEED + seed_show,
    )
global_meta_s.fit(X_meta_train_s, y_meta_train_s)

def score_meta_fn_show(Xa, yc):
    out = np.zeros(len(Xa), dtype=np.float64)
    for i in range(len(Xa)):
        cls = int(yc[i])
        mdl = models_by_cls_s.get(cls, global_meta_s)
        out[i] = mdl.predict_proba(Xa[i:i+1])[:, 1][0]
    return out

def score_lira_fn_show(Xa, yc):
    p_true = np.clip(Xa[:, 7], 1e-6, 1 - 1e-6)
    s_vals = np.log(p_true / (1.0 - p_true))
    out = np.zeros(len(Xa), dtype=np.float64)
    for i in range(len(Xa)):
        prm = lira_params[int(yc[i])]
        li = normal_logpdf(s_vals[i], prm['mu_in'], prm['sd_in'])
        lo = normal_logpdf(s_vals[i], prm['mu_out'], prm['sd_out'])
        out[i] = li - lo
    return out

score_meta_val_s = score_meta_fn_show(Xa_val_s, yc_val_s)
score_lira_val_s = score_lira_fn_show(Xa_val_s, yc_val_s)
score_meta_te_s = score_meta_fn_show(Xa_te_s, yc_te_s)
score_lira_te_s = score_lira_fn_show(Xa_te_s, yc_te_s)

candidates_val = {
    'meta_only': rank01(score_meta_val_s),
    'lira_only': rank01(score_lira_val_s),
    'boundary_only': rank01(sb_val_s),
    'fusion_0.45_0.25_0.30': 0.45 * rank01(score_meta_val_s) + 0.25 * rank01(score_lira_val_s) + 0.30 * rank01(sb_val_s),
    'fusion_0.35_0.45_0.20': 0.35 * rank01(score_meta_val_s) + 0.45 * rank01(score_lira_val_s) + 0.20 * rank01(sb_val_s),
    'fusion_0.30_0.30_0.40': 0.30 * rank01(score_meta_val_s) + 0.30 * rank01(score_lira_val_s) + 0.40 * rank01(sb_val_s),
    'fusion_0.40_0.35_0.25': 0.40 * rank01(score_meta_val_s) + 0.35 * rank01(score_lira_val_s) + 0.25 * rank01(sb_val_s),
    'fusion_0.50_0.30_0.20': 0.50 * rank01(score_meta_val_s) + 0.30 * rank01(score_lira_val_s) + 0.20 * rank01(sb_val_s),
    'fusion_0.25_0.50_0.25': 0.25 * rank01(score_meta_val_s) + 0.50 * rank01(score_lira_val_s) + 0.25 * rank01(sb_val_s),
}

best_name_s = max(candidates_val, key=lambda k: roc_auc_score(ya_val_s, candidates_val[k]))

if best_name_s == 'meta_only':
    score_show = rank01(score_meta_te_s)
elif best_name_s == 'lira_only':
    score_show = rank01(score_lira_te_s)
elif best_name_s == 'boundary_only':
    score_show = rank01(sb_te_s)
elif best_name_s == 'fusion_0.45_0.25_0.30':
    score_show = 0.45 * rank01(score_meta_te_s) + 0.25 * rank01(score_lira_te_s) + 0.30 * rank01(sb_te_s)
elif best_name_s == 'fusion_0.35_0.45_0.20':
    score_show = 0.35 * rank01(score_meta_te_s) + 0.45 * rank01(score_lira_te_s) + 0.20 * rank01(sb_te_s)
elif best_name_s == 'fusion_0.40_0.35_0.25':
    score_show = 0.40 * rank01(score_meta_te_s) + 0.35 * rank01(score_lira_te_s) + 0.25 * rank01(sb_te_s)
elif best_name_s == 'fusion_0.50_0.30_0.20':
    score_show = 0.50 * rank01(score_meta_te_s) + 0.30 * rank01(score_lira_te_s) + 0.20 * rank01(sb_te_s)
elif best_name_s == 'fusion_0.25_0.50_0.25':
    score_show = 0.25 * rank01(score_meta_te_s) + 0.50 * rank01(score_lira_te_s) + 0.25 * rank01(sb_te_s)
else:
    score_show = 0.30 * rank01(score_meta_te_s) + 0.30 * rank01(score_lira_te_s) + 0.40 * rank01(sb_te_s)

pred_show = (score_show >= 0.5).astype(int)

# Vrais membres correctement détectés membres.
stolen_ok_mask = (ya_te_s == 1) & (pred_show == 1)

if stolen_ok_mask.sum() == 0:
    print("Aucun exemple 'volé correctement' pour ce seed.")
else:
    idx_ok = idx_te_s[stolen_ok_mask]
    raw_ok = X_query_target_raw[idx_ok]
    score_ok = score_show[stolen_ok_mask]

    stolen_ok_df = pd.DataFrame(raw_ok, columns=FEATURE_COLS)
    stolen_ok_df["shadow_membership_score"] = score_ok
    stolen_ok_df["true_member"] = 1
    stolen_ok_df["pred_member"] = 1
    stolen_ok_df = stolen_ok_df.sort_values("shadow_membership_score", ascending=False).head(show_n)

    print(f"=== Portion de données volées correctement (top {min(show_n, len(stolen_ok_df))}) ===")
    print(f"Seed affiché: {seed_show} | Scoreur sélectionné: {best_name_s}")
    display(stolen_ok_df.round(4))

=== Portion de données volées correctement (top 10) ===
Seed affiché: 55 | Scoreur sélectionné: boundary_only


,MR Delay,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF,shadow_membership_score,true_member,pred_member
6,1598.0,83.0,16.0,2.0,29.0,0.0,1323.0,0.718,1.327,1.0000,1,1
5,1603.0,85.0,12.0,4.0,30.0,0.0,1699.0,0.705,1.033,0.9796,1,1
29,737.0,80.0,18.0,1.0,30.0,0.0,1436.0,0.663,1.222,0.9592,1,1
10,952.0,74.0,18.0,2.0,30.0,0.0,1400.0,0.752,1.254,0.9184,1,1
1,729.0,72.0,11.0,4.0,21.0,1.0,1489.0,0.686,1.179,0.8980,1,1
4,754.0,98.0,17.0,1.0,21.0,2.0,1503.0,0.660,1.168,0.8571,1,1
14,1345.0,76.0,18.0,2.0,30.0,0.0,1550.0,0.758,1.133,0.8469,1,1
0,1695.0,81.0,16.0,3.0,30.0,0.0,1836.0,0.744,0.956,0.8367,1,1
16,446.0,88.0,12.0,3.0,30.0,0.0,1445.0,0.719,1.215,0.8265,1,1
34,504.0,77.0,16.0,3.0,16.0,1.0,1590.0,0.696,1.104,0.8163,1,1


In [11]:
# 7) Tableau final comparatif

attack_metrics = pd.DataFrame([
    thr_metrics,
    lr_metrics,
    shadow_metrics,
]).sort_values("auc", ascending=False)

print("=== Métriques comparatives MIA ===")
display(attack_metrics.round(4))

best_attack = attack_metrics.iloc[0]
print(f"Best MIA: {best_attack['attack']} | AUC={best_attack['auc']:.4f}")

=== Métriques comparatives MIA ===


,attack,auc,accuracy,precision,recall,f1
2,shadow_meta,0.6151,0.6485,0.6807,0.6815,0.6802
0,threshold_loss,0.4821,0.3099,0.2222,1.0000,0.3636
1,logistic,0.4818,0.8028,0.0000,0.0000,0.0000


Best MIA: shadow_meta | AUC=0.6151
